# Environment

In [62]:
import sys
print(sys.executable)
#check_loc = !which pytesseract
#print(check_loc)

/opt/homebrew/anaconda3/bin/python


In [63]:
#conda install -c conda-forge tesseract --solver=classic

In [64]:
import os
import sys
import subprocess
import pandas as pd
import uuid
import re

# Required packages for the project
required_packages = [
    "pytesseract", "pdf2image", "pillow", "pymupdf", "pandas", "python-docx"
]

def check_and_install_packages():
    """
    Checks if all required packages are installed. If not, installs them.
    """
    print(" Checking required packages...")
    
    # Get a list of installed packages
    installed_packages = {pkg.split("==")[0].lower() for pkg in subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout.split("\n") if pkg}



    missing_packages = [pkg for pkg in required_packages if pkg.lower() not in installed_packages]

    if not missing_packages:
        print(" All required packages are installed!")
    else:
        print(f" Missing packages detected: {missing_packages}")
        for pkg in missing_packages:
            print(f" Installing {pkg}...")
            subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(" Installation complete!")

def check_pwd(expected_path=None):
    """
    Checks if the current working directory (PWD) is correct.
    If an expected path is provided, it will compare against the current directory.
    """
    current_path = os.getcwd()
    print(f" Current Working Directory: {current_path}")

    if expected_path:
        if os.path.abspath(current_path) != os.path.abspath(expected_path):
            print(f" Warning: Expected to be in {expected_path}, but found {current_path}.")
        else:
            print(" PWD is correct.")

# Run the checks
check_and_install_packages()
check_pwd()  # Optionally, pass an expected path e.g., check_pwd("/Users/emilycastelan/FeedbackProject")


🔍 Checking required packages...
⚠️ Missing packages detected: ['pillow', 'pandas']
 Installing pillow...
 Installing pandas...
 Installation complete!
 Current Working Directory: /Users/emilycastelan/Desktop/Census/FeedbackProject


In [65]:
# Environment: 
#pip install pytesseract pdf2image pillow
import sys

# Import required libraries
#!pip install pytesseract
import pytesseract
# !brew install tesseract
from PIL import Image
import io

#!brew install tesseract
#!tesseract --version


#pip install pymupdf
#from pymupdf import fitz  # PyMuPDF
import pandas as pd
import re

import pandas as pd
import uuid
import re

# Read in files
#!pip install pymupdf
import fitz
#!pip install python-docx
#conda install -c conda-forge tesseract
from docx import Document
#print("Tesseract Version:", pytesseract.get_tesseract_version())

# PRODUCTS

1. CQAS: Not product specific, submitted emails  
2. Crosswalk: Not product specific, describes links between different products and features between 2010 and 2020 products. 
3. DHC A: Detailed Demographic and Housing Characteristics File A. Provides sex-by-age statistics for racial and ethnic groups.
4. DHC B: Detailed Demographic and Housing Characteristics File B. Provides data on household tenure for the same races and ethnicities as DHC A. 
5. DHC: Demographic and Housing Characteristics. Includes the same 6 tables in PL, but more expansive. 
6. PL: Redistricting Data (public law 94-171). Specific to voting rights. 
7. PPMF: Privacy-Protected Microdata. Individual records that tabulate PL and DHC.
8. Product S-DHC: Supplemental Demographic and Housing Characteristics File, includes household type and family data.


## Features to Extract: 
1. ID
2. Name
3. Organization
4. date
5. product(s)
6. file input type/category
7. issue
8. solution suggested
9. sentiment score
10. frequency of concern(s)
11. themes = 
    - accuracy of data
    - privacy 
    - geographic mention
    - community mention 
14. related product/use 
15. status of resolution


In [66]:
# Variations of names


# Define product variations
PRODUCT_KEYWORDS = {
    "CQAS": ["CQAS", "Controlled Correspondence"],
    "Crosswalk": ["Crosswalk"],
    "DHC A": ["DHC A", "DHC_A", " Detailed Demographic Housing Characteristics",
    "sex by age", "file A", "Detailed Demographic and Housing Characteristics"],
    "DHC B": ["DHC B", "DHC_B",
     "Detailed Demographic Housing Characteristics tenure", "Household tenure",
     "file B"],
    "DHC": ["DHC", "Demographic Housing Characteristics", 
    "Demographic and Housing Characteristics"],  # Ensuring no typos
    "PL": ["PL", "Public Law 94-171", "Public Law",  "voting rights",
    "Redistricting"],
    "PPMF": ["PPMF", "Privacy-Protected Microdata"],
    "Product S-DHC": ["DHC S", "Product S-DHC", "Supplemental DHC",
    "Holdhold type", "Supplemental Demographic Housing Characteristics",
    "Supplemental Demographic and Housing Characteristics"
    ]
}



## Dataframe

In [67]:
# Dataframe
# Re-import necessary libraries
# Define the structured DataFrame with the specified features
df_feedback_frame = pd.DataFrame(columns=[
    "ID", "Name", "Incoming Feedback", "Organization", "Date", "Product(s)", "File Input Type/Category", 
    "Issue", "Solution Suggested", "Sentiment Score", "Frequency of Concern(s)", 
    "Themes", "Related Product/Use", "Status of Resolution"
])

# Function to generate unique IDs
def generate_unique_id():
    return str(uuid.uuid4())

# Example: Adding placeholder rows (can be replaced with actual processing later)
#df.loc[0] = [generate_unique_id(), None, None, "2024-01-01", None, None, None, None, None, None, None, None, None]
#df.loc[1] = [generate_unique_id(), None, None, "2024-01-02", None, None, None, None, None, None, None, None, None]


## Identifying products function

In [68]:
def tag_products(feedback_text):
    """
    Identifies products mentioned in feedback and returns a list of tags.
    """
    feedback_text = feedback_text.lower()  # Normalize text for case-insensitive matching
    tags = set()  # Use a set to avoid duplicate tags

    for product, variations in PRODUCT_KEYWORDS.items():
        for variation in variations:
            # Using regex for partial word matching, e.g., "DHC A" inside "dhc a table"
            if re.search(rf"\b{re.escape(variation.lower())}\b", feedback_text):
                tags.add(product)

    return list(tags)

# Example usage:
feedback_example = "The Crosswalk between 2010 and 2020 was really helpful, but I had issues with the DHC B data."
print(tag_products(feedback_example))

['DHC', 'Crosswalk', 'DHC B']


# READ IN

In [69]:
# Paths 
#diff_priv_path = "Differential Privacy and Disclosure Avoidance Revised DP Communications Report by Hampton Wilson for 2022.pdf"
#FSCPE_path = "FSCPE Steering Committee_CQAS-11795signed by Director to Jan Vink Nov 22, 2022.pdf"
#Crosswalk_path = Document("Crosswalk Detailed DHC Feedback_with Congressional_7 page doc_Received Post Deadline of 12 18 2021.docx")

diff_priv_path = "Differential Privacy and Disclosure Avoidance Revised DP Communications Report by Hampton Wilson for 2022.pdf"

diff_priv_pdf = fitz.open(diff_priv_path)

## Fq Detection

In [70]:

from collections import Counter
import re

# Define product variations
PRODUCT_KEYWORDS = {
    "CQAS": ["CQAS"],
    "Crosswalk": ["Crosswalk"],
    "DHC A": ["DHC A", "DHC_A", "Detailed Demographic Housing Characteristics", "sex by age"],
    "DHC B": ["DHC B", "DHC_B", "Detailed Demographic Housing Characteristics tenure", "Household tenure"],
    "DHC": ["DHC", "Demographic Housing Characteristics"],
    "PL": ["PL", "Public Law 94-171", "Public Law", "voting rights"],
    "PPMF": ["PPMF", "Privacy-Protected Microdata"],
    "Product S-DHC": ["DHC S", "Product S-DHC", "Supplemental DHC",
                      "Household type", "Supplemental Demographic Housing Characteristics"]
}

# Initialize a counter for product mentions
product_frequency = Counter()

# Iterate through the feedback text and count occurrences of each product keyword
for feedback in df_feedback_frame["Incoming Feedback"]:
    feedback_lower = feedback.lower()  # Normalize text for case-insensitive matching
    
    for product, keywords in PRODUCT_KEYWORDS.items():
        for keyword in keywords:
            if re.search(rf"\b{re.escape(keyword.lower())}\b", feedback_lower):
                product_frequency[product] += 1

# Convert frequency counts to a DataFrame for better visualization
#df_product_frequency = pd.DataFrame(product_frequency.items(), columns=["Product", "Frequency"])

# Display the product frequency DataFrame
#df_product_frequency

#######################################

# PART 1: EXTRACT FULL FEEDBACK TEXT
# Extracting and Merging Feedback Text from PDFs with OCR

This script processes a PDF containing feedback requests by extracting text from both the document's text layer and any embedded images using Optical Character Recognition (OCR). The extraction is performed in multiple steps to ensure complete data retrieval, including handling cases where text is embedded in images rather than directly within the PDF text layer.

## Workflow Overview

The script follows a structured approach:

1. **Extract Text-Based Feedback Entries**  
   - Parses the PDF document to identify structured feedback entries.  
   - Captures text spanning multiple pages while tracking entry page ranges.  
   - Detects missing text by flagging entries that end with the word "Incoming" but lack further content.  
   - Inserts a placeholder (`[IMAGE_TEXT_HERE]`) in cases where text appears to be missing, likely due to being embedded in an image.  

2. **Extract Text from Images Separately using OCR**  
   - Iterates through each page of the PDF, identifying and extracting images.  
   - Applies Optical Character Recognition (OCR) via Tesseract to retrieve any text present within these images.  
   - Stores extracted text, associating it with the corresponding page number.  

3. **Merge OCR-Extracted Text into Flagged Feedback Entries**  
   - Identifies feedback entries containing the `[IMAGE_TEXT_HERE]` placeholder.  
   - Matches the OCR-extracted text to the correct feedback entries based on page numbers.  
   - Replaces the placeholder with the extracted OCR text.  

4. **Save and Verify the Final Output**  
   - Exports the processed data as a CSV file for further review.  
   - Prints a summary of feedback entries where OCR text was successfully merged, ensuring completeness.  

This process ensures that all feedback is accurately captured, including cases where text was embedded in images. The final output provides a cleaned and merged dataset, making it suitable for further analysis.


In [71]:
# MODIFIED PT 1 FOR RESPONSE EXTRACTION

def extract_full_feedback_with_response(pdf_path):
    """
    Extracts full request entries including 'Response' from the PDF and tracks page ranges.
    """
    doc = fitz.open(pdf_path)
    feedback_entries = []
    current_entry_text = []
    current_response_text = []
    collecting_request = False
    collecting_response = False
    entry_start_page = None
    response_start_page = None

    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        lines = text.split("\n")
        images = page.get_images(full=True)

        print(f" Page {page_num + 1}: Found {len(images)} image(s)")

        for line in lines:
            line = line.strip()

            # Start collecting when we see "Requestor’s Name:"
            if "Requestor’s Name:" in line:
                if current_entry_text or current_response_text:
                    feedback_entries.append({
                        "request_text": "\n".join(current_entry_text),
                        "response_text": "\n".join(current_response_text),
                        "start_page": entry_start_page,
                        "end_page": page_num
                    })
                current_entry_text = [line]
                current_response_text = []
                entry_start_page = page_num
                collecting_request = True
                collecting_response = False
                continue

            # Start collecting response when we see "Response"
            if "Response" in line:
                collecting_request = False
                collecting_response = True
                response_start_page = page_num
                continue

            # Stop collecting when we reach another "Requestor’s Name:"
            if "Requestor’s Name:" in line:
                collecting_request = False
                collecting_response = False
                continue

            # Capture text
            if collecting_request:
                current_entry_text.append(line)
            elif collecting_response:
                current_response_text.append(line)

        # **New Robust Check for Missing Text After "Incoming"**
        if any("Incoming" in l for l in current_entry_text[-3:]):  # Check if "Incoming" appears in last 3 lines
            last_lines = [l.strip() for l in current_entry_text[-3:] if l.strip()]
            incoming_index = next((i for i, l in enumerate(last_lines) if "Incoming" in l), None)

            if incoming_index is not None and (incoming_index == len(last_lines) - 1 or last_lines[incoming_index + 1] == ""):
                current_entry_text.append("[IMAGE_TEXT_HERE]")  # NEW FLAG FOR MERGING

    if current_entry_text or current_response_text:
        feedback_entries.append({
            "request_text": "\n".join(current_entry_text),
            "response_text": "\n".join(current_response_text),
            "start_page": entry_start_page,
            "end_page": page_num
        })

    df_text = pd.DataFrame(feedback_entries)

    # Print flagged entries to verify correct placement
    flagged_entries = df_text[df_text["request_text"].str.contains("\[IMAGE_TEXT_HERE\]", na=False)]
    print(f"📌 Flagged {len(flagged_entries)} feedback entries with missing text:\n")
    print(flagged_entries[["start_page", "end_page", "request_text"]].head(10))

    return df_text


In [72]:
def extract_full_feedback_with_response(pdf_path):
    """
    Extracts full request entries including 'Response' from the PDF and tracks page ranges,
    while keeping 'Response' as a subtitle to allow future parsing.
    """
    doc = fitz.open(pdf_path)
    feedback_entries = []
    current_entry_text = []
    collecting = False  
    entry_start_page = None  

    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        lines = text.split("\n")
        images = page.get_images(full=True)  

        print(f" Page {page_num + 1}: Found {len(images)} image(s)")

        for line in lines:
            line = line.strip()

            # Start collecting when we see "Requestor’s Name:"
            if "Requestor’s Name:" in line:
                if current_entry_text:
                    feedback_entries.append({
                        "text": "\n".join(current_entry_text),  # Keep everything together
                        "start_page": entry_start_page,
                        "end_page": page_num
                    })
                current_entry_text = [line]  # Start new entry
                entry_start_page = page_num
                collecting = True
                continue

            # Keep "Response" inside the extracted text
            if "Response" in line:
                collecting = True  # Do not stop collecting—keep it in the text
                current_entry_text.append("\n" + line)  # Preserve subtitle formatting
                continue

            # Stop collecting only when another "Requestor’s Name:" appears
            if "Requestor’s Name:" in line:
                collecting = False
                continue

            # Capture text
            if collecting:
                current_entry_text.append(line)

        # **New Robust Check for Missing Text After "Incoming"**
        if any("Incoming" in l for l in current_entry_text[-3:]):  
            last_lines = [l.strip() for l in current_entry_text[-3:] if l.strip()]
            incoming_index = next((i for i, l in enumerate(last_lines) if "Incoming" in l), None)

            if incoming_index is not None and (incoming_index == len(last_lines) - 1 or last_lines[incoming_index + 1] == ""):
                current_entry_text.append("[IMAGE_TEXT_HERE]")  # Flag for missing image text

    # Save the last extracted entry
    if current_entry_text:
        feedback_entries.append({
            "text": "\n".join(current_entry_text),
            "start_page": entry_start_page,
            "end_page": page_num
        })

    df_text = pd.DataFrame(feedback_entries)

    # Print flagged entries to verify correct placement
    flagged_entries = df_text[df_text["text"].str.contains("\[IMAGE_TEXT_HERE\]", na=False)]
    print(f" Flagged {len(flagged_entries)} feedback entries with missing text:\n")
    print(flagged_entries[["start_page", "end_page", "text"]].head(10))  

    return df_text


In [73]:
# PART 2
def extract_text_from_images_separately(pdf_path):
    """
    Extracts text from images in the PDF and groups them by feedback page ranges.
    """
    doc = fitz.open(pdf_path)
    image_texts = {}  # Dictionary to store OCR text grouped by page

    for page_num, page in enumerate(doc):
        images = page.get_images(full=True)
        if not images:
            continue  

        print(f" Page {page_num + 1}: Found {len(images)} image(s)")

        extracted_text = []
        for idx, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)

            if not base_image:
                print(f"⚠️ Failed to extract image on page {page_num + 1}")
                continue

            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))

            # Apply OCR
            text = pytesseract.image_to_string(image).strip()
            extracted_text.append(text)

        if extracted_text:
            image_texts[page_num] = "\n".join(extracted_text)  # Store OCR text per page  

    return image_texts


In [74]:
# PART 3

def merge_ocr_text_into_missing_feedback(df_text, ocr_texts):
    """
    Merges OCR text into feedback text by matching page ranges.
    """
    updated_entries = []

    print(" Checking for flagged image text areas to replace with OCR text...")
    print(f" Available OCR pages: {list(ocr_texts.keys())}")  

    for index, row in df_text.iterrows():
        entry_text = row["text"]
        start_page, end_page = row["start_page"], row["end_page"]

        #  Gather all OCR text from the range of pages this feedback spans
        relevant_ocr_texts = []
        for page in range(start_page, end_page + 1):
            if page in ocr_texts:
                relevant_ocr_texts.append(ocr_texts[page])

        #  If we found OCR text, replace flag
        if "[IMAGE_TEXT_HERE]" in entry_text and relevant_ocr_texts:
            merged_ocr_text = "\n".join(relevant_ocr_texts)
            updated_text = entry_text.replace("[IMAGE_TEXT_HERE]", f" OCR Extracted Text:\n{merged_ocr_text}")
            updated_entries.append(updated_text)
            print(f" Merged OCR text into Entry {index} spanning pages {start_page}–{end_page}")
        else:
            updated_entries.append(entry_text)

    df_text["text"] = updated_entries  # Update DataFrame column  

    print(f" DataFrame AFTER merging:\n{df_text[df_text['text'].str.contains('OCR Extracted Text', na=False)].head()}")  
    return df_text


In [75]:
#  Step 1: Extract feedback text with placeholder flags
df_feedback_text = extract_full_feedback_with_response("Differential Privacy and Disclosure Avoidance Revised DP Communications Report by Hampton Wilson for 2022.pdf")

# Step 2: Extract text from images separately
image_text_dict = extract_text_from_images_separately("Differential Privacy and Disclosure Avoidance Revised DP Communications Report by Hampton Wilson for 2022.pdf")

# Step 3: Merge OCR text into feedback text where missing
df_final = merge_ocr_text_into_missing_feedback(df_feedback_text, image_text_dict)

# Step 4: Save and verify
df_final.to_csv("final_with_images.csv", index=False)
print(" Saved CSV. Checking OCR-merged contents...")

ocr_entries = df_final[df_final["text"].str.contains("OCR Extracted Text", na=False)]
print(f" Found {len(ocr_entries)} rows with OCR text:")
print(ocr_entries.head())  # Verify OCR-mapped rows


📄 Page 1: Found 0 image(s)
📄 Page 2: Found 0 image(s)
📄 Page 3: Found 0 image(s)
📄 Page 4: Found 0 image(s)
📄 Page 5: Found 0 image(s)
📄 Page 6: Found 0 image(s)
📄 Page 7: Found 0 image(s)
📄 Page 8: Found 0 image(s)
📄 Page 9: Found 0 image(s)
📄 Page 10: Found 0 image(s)
📄 Page 11: Found 0 image(s)
📄 Page 12: Found 0 image(s)
📄 Page 13: Found 0 image(s)
📄 Page 14: Found 0 image(s)
📄 Page 15: Found 0 image(s)
📄 Page 16: Found 0 image(s)
📄 Page 17: Found 0 image(s)
📄 Page 18: Found 2 image(s)
📄 Page 19: Found 0 image(s)
📄 Page 20: Found 0 image(s)
📄 Page 21: Found 1 image(s)
📄 Page 22: Found 1 image(s)
📄 Page 23: Found 1 image(s)
📄 Page 24: Found 1 image(s)
📄 Page 25: Found 2 image(s)
📄 Page 26: Found 1 image(s)
📄 Page 27: Found 2 image(s)
📄 Page 28: Found 1 image(s)
📄 Page 29: Found 0 image(s)
📄 Page 30: Found 0 image(s)
📄 Page 31: Found 0 image(s)
📄 Page 32: Found 0 image(s)
📄 Page 33: Found 0 image(s)
📄 Page 34: Found 0 image(s)
📄 Page 35: Found 1 image(s)
📄 Page 36: Found 1 image(s)
📄

In [76]:
sanity_check = df_final
sanity_check.to_csv("sanity_check.csv")

# PART 2: PARSE THE FEEDBACK EXTRACTED

In [77]:
import re
import pandas as pd
from datetime import datetime
from dateutil import parser  # Helps with flexible date parsing

def parse_feedback_from_df(df_final):
    """
    Parses extracted feedback text into structured columns:
    - CQAS No
    - Requestor's Name
    - Date of Request
    - Incoming Feedback
    - Response
    """
    parsed_entries = []

    # Define regex patterns
    cqas_pattern = r"CQAS[-–—]?\s*(\d{3,6})"  # Handles variations of CQAS format
    date_pattern = r"\b(?:\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2}, \d{4}|\d{4}-\d{2}-\d{2})\b"
    incoming_pattern = r"Incoming\s*(.*?)(?=(Response|Requestor’s Name:|$))"
    response_pattern = r"Response\s*(.*?)("

    prompt_keywords = ["CQAS", "Requestor’s Name:", "Date of Request:", "Incoming", "Response"]

    for index, row in df_final.iterrows():
        raw_text = row["text"]  # Extract the full raw text entry
        lines = raw_text.split("\n")  # Split into individual lines

        # Debugging first few rows
        if index < 9:
            print(f"\n **Row {index} - Raw Text:**\n{raw_text}")
            print("=" * 80)

        # 1 **Extract CQAS Number**
        cqas_match = re.search(cqas_pattern, raw_text)
        cqas_no = cqas_match.group(1) if cqas_match else "MISSING_CQAS"

        # 2️ **Extract Date (Flexible Parsing)**
        date_match = re.search(date_pattern, raw_text)
        date_str = date_match.group(0) if date_match else None

        # Debug extracted dates
        if 15 < index < 19:
            print(f" **Row {index} - Extracted Date String:** {date_str}")

        parsed_date = None
        if date_str:
            try:
                parsed_date = parser.parse(date_str).date()  # Auto-detects format
            except ValueError:
                parsed_date = None  # If invalid, set as None

        # 3️ **Extract Feedback (Everything After "Incoming")**
        feedback_match = re.search(incoming_pattern, raw_text, re.DOTALL)
        feedback_text = feedback_match.group(1).strip() if feedback_match else "MISSING_FEEDBACK"

        # 4️ **Extract Requestor Name**
        requestor_name = "MISSING_NAME"
        possible_names = []

        # Iterate through lines to find a valid name
        for line in lines:
            line = line.strip()

            # Skip known headers/prompts
            if any(prompt in line for prompt in prompt_keywords):
                continue

            # Exclude CQAS, date, and empty lines
            if line and line != cqas_no and line != date_str and not re.match(r".*[:：]$", line):
                possible_names.append(line)

        if possible_names:
            requestor_name = possible_names[0]  # Use the first non-header line as the name

        # Debug problematic entries
        if index < 9:
            print(f" **Row {index} - Extracted Fields:**")
            print(f"CQAS: {cqas_no}, Name: {requestor_name}, Date: {parsed_date}, Feedback: {feedback_text}\n")
            print("=" * 80)

        # Append structured data
        parsed_entries.append({
            "CQAS No": cqas_no,
            "Requestor’s Name": requestor_name,
            "Date of Request": parsed_date,
            "Incoming Feedback": feedback_text
        })

    # Convert to DataFrame
    df_parsed = pd.DataFrame(parsed_entries)

    return df_parsed


### Modified Parse for Response

In [78]:
import re
import pandas as pd
from datetime import datetime
from dateutil import parser  # Helps with flexible date parsing

def parse_feedback_from_df(df_final):
    """
    Parses extracted feedback text into structured columns:
    - CQAS No
    - Requestor's Name
    - Date of Request
    - Incoming Feedback
    - Response
    """
    parsed_entries = []

    # Define regex patterns
    cqas_pattern = r"CQAS\s*[-–—]?\s*(\d{3,6})"  # Handles variations of CQAS format
    date_pattern = r"\b(?:\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2}, \d{4}|\d{4}-\d{2}-\d{2})\b"
    incoming_pattern = r"Incoming\s*(.*?)(?=\bResponse\b|Requestor’s Name:|$)"  # 🔥 Modified to stop at 'Response'
    response_pattern = r"Response\s*(.*?)(?=Requestor’s Name:|$)"  # 🔥 New pattern to extract response

    prompt_keywords = ["CQAS", "Requestor’s Name:", "Date of Request:", "Incoming", "Response"]

    for index, row in df_final.iterrows():
        raw_text = row["text"]  # Extract the full raw text entry
        lines = raw_text.split("\n")  # Split into individual lines

        # Debugging first few rows
        if index < 9:
            print(f"\n **Row {index} - Raw Text:**\n{raw_text}")
            print("=" * 80)

        # 1 **Extract CQAS Number**
        cqas_match = re.search(cqas_pattern, raw_text)
        cqas_no = cqas_match.group(1) if cqas_match else "MISSING_CQAS"

        # 2 **Extract Date (Flexible Parsing)**
        date_match = re.search(date_pattern, raw_text)
        date_str = date_match.group(0) if date_match else None

        # Debug extracted dates
        if 15 < index < 19:
            print(f" **Row {index} - Extracted Date String:** {date_str}")

        parsed_date = None
        if date_str:
            try:
                parsed_date = parser.parse(date_str).date()  # Auto-detects format
            except ValueError:
                parsed_date = None  # If invalid, set as None

        # 3️ **Extract Incoming Feedback**
        incoming_match = re.search(incoming_pattern, raw_text, re.DOTALL)
        incoming_text = incoming_match.group(1).strip() if incoming_match else "MISSING_FEEDBACK"

        # 4️ **Extract Response**
        response_match = re.search(response_pattern, raw_text, re.DOTALL)
        response_text = response_match.group(1).strip() if response_match else "MISSING_RESPONSE"

        # 5️ **Extract Requestor Name**
        requestor_name = "MISSING_NAME"
        possible_names = []

        # Iterate through lines to find a valid name
        for line in lines:
            line = line.strip()

            # Skip known headers/prompts
            if any(prompt in line for prompt in prompt_keywords):
                continue

            # Exclude CQAS, date, and empty lines
            if line and line != cqas_no and line != date_str and not re.match(r".*[:：]$", line):
                possible_names.append(line)

        if possible_names:
            requestor_name = possible_names[0]  # Use the first non-header line as the name

        # Debug problematic entries
        if index < 9:
            print(f" **Row {index} - Extracted Fields:**")
            print(f"CQAS: {cqas_no}, Name: {requestor_name}, Date: {parsed_date}, Incoming: {incoming_text}, Response: {response_text}\n")
            print("=" * 80)

        # Append structured data
        parsed_entries.append({
            "CQAS No": cqas_no,
            "Requestor’s Name": requestor_name,
            "Date of Request": parsed_date,
            "Incoming Feedback": incoming_text,
            "Response": response_text  # NEW COLUMN
        })

    # Convert to DataFrame
    df_parsed = pd.DataFrame(parsed_entries)

    return df_parsed


In [79]:
df_parsed_feedback = parse_feedback_from_df(df_final)
df_parsed_feedback.head(22)



 **Row 0 - Raw Text:**
Requestor’s Name:
Toni C. Atkins, Senate President pro Tempore

Anthony Rendon, Speaker of the Assembly
Date of Request:
3/01/21


CQAS No:
CQAS-10963

Incoming

Dear Mr. Klain: We write today on a key issue affecting the 2020 U.S. Census. The Trump Administration efforts to
manipulate the Census are well known to all of us. Thankfully President Biden has been able to halt the most
obvious manipulation - the adjustment affecting non-citizens. However, there is still another adjustment developed
by the Trump Administration that will affect the accuracy of the Census and the ability to ensure both fair
representation of minority communities and fair distribution of federal resources. The adjustment to Census data
that remains a concern is referred to as "differential privacy." It has been developed by the Census Bureau as part of
its mandate to maintain the confidentiality of individual American residents, while at the same time producing
accurate detailed data. T

,CQAS No,Requestor’s Name,Date of Request,Incoming Feedback,Response
0,10963,"Toni C. Atkins, Senate President pro Tempore",2021-03-01,Dear Mr. Klain: We write today on a key issue ...,"Thank you for your letter to Ronald Klain, Whi..."
1,10822,Wendy Underhill,2020-12-22,"NCSL wrote to the bureau on May 26, 2020 expre...","As you pointed out in your message, the U.S. C..."
2,10611,"Letter from A. Donald McEachin, Jamie Raskin a...",2020-09-21,We write to express concern with the U.S. Cens...,"In your letter, you reference some concerns fr..."
3,10337,National Conference of State Legislators,2020-05-06,Dear Director Dillingham: On behalf of the nat...,The U.S. Census Bureau is very grateful for an...
4,10336,National States Geographic Information Council,2020-05-21,Date 504.265.9720 www.nsgic.org info@nsgic.org...,"As you note in your letter, decennial census d..."
5,10151,Differential Privacy City of Alexander,2020-02-03,RE: Effect of U.S. Census Differenti Privacy o...,Thank you for your letter regarding the effect...
6,09649,"Kevin Allis, National Congress of American Ind...",2019-07-24,We are writing to share our concerns about the...,Thank you for your letter regarding the U.S. C...
7,10135,Angela Hallowell & Amanda Rector,2020-02-02,"The Office of the State Economist, within the ...",MISSING_RESPONSE
8,11109,Dante Desiderio,2021-05-21,We write to provide comments in response to th...,No response provided
9,10975,"Jamie Gomez, Chief of Staff, NCAI",2021-03-08,We write to provide comments in response to th...,MISSING_RESPONSE


In [80]:
df_parsed_feedback.to_csv("DF with responses.csv")